In [ ]:
%pip install torch pillow transformers==4.50 accelerate pandas

In [ ]:
import os
import re
import torch
import pandas as pd
from PIL import Image

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto"
)

print("Model loaded. CUDA:", torch.cuda.is_available())

In [ ]:
from huggingface_hub import snapshot_download

repo_id = "ReadingTimeMachine/visual_qa_histograms"

local_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["example_hists_larger/**"]
)

print(local_dir)

In [ ]:
img_dir = os.path.join(local_dir, "example_hists_larger", "imgs")

print(img_dir)
print(os.listdir(img_dir)[:5])

In [ ]:
df1 = pd.read_csv("hist_complex_parsed.csv")
df2 = pd.read_csv("hist_larger_parsed.csv")

df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)
print(df.columns)
df.head()

In [ ]:
print("CSV path example:", df.iloc[0]["image_path"])
print("Extracted filename:", os.path.basename(df.iloc[0]["image_path"]))
print("Actual image files:", os.listdir(img_dir)[:5])

In [ ]:
import re

def fix_filename(p):
    fname = os.path.basename(p)  # ima_000123.jpeg
    
    # extract the number (000123)
    match = re.search(r"(\d+)", fname)
    
    if match:
        num = match.group(1)
        return f"id_{num}.jpeg"
    else:
        return None

df["image_file"] = df["image_path"].apply(fix_filename)
df["full_image_path"] = df["image_file"].apply(lambda f: os.path.join(img_dir, f) if f else None)

df["exists"] = df["full_image_path"].apply(lambda p: os.path.exists(p) if p else False)

print(df["exists"].value_counts())

In [ ]:
df_valid = df[df["exists"]]
print(df_valid.shape)
df_valid.head()

In [ ]:
row = df_valid.iloc[0]

print(row["full_image_path"])

image = Image.open(row["full_image_path"]).convert("RGB")
image

In [ ]:
PROMPTS = {
    "zero_shot": lambda stat: f"What is the {stat} of this histogram?",
    "structured": lambda stat: (
        f"What is the {stat} of this histogram? "
        f"If the value is not explicitly labeled, estimate carefully from the axis. "
        f"Give your answer as a number."
    ),
    "chain_of_thought": lambda stat: (
        f"What is the {stat} of this histogram? "
        f"First identify the axis range, then locate the relevant part of the histogram, "
        f"then estimate the value. Give the final answer as a number."
    ),
}

In [ ]:
def ask_model(image, question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120
        )

    input_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][input_len:]
    response = processor.decode(generated_ids, skip_special_tokens=True)
    return response

In [ ]:
import re

def extract_number(text):
    if text is None:
        return None

    text = str(text).replace(",", "")
    matches = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)

    if not matches:
        return None

    try:
        return float(matches[0])
    except ValueError:
        return None

In [ ]:
from PIL import Image

row = df_valid.iloc[0]
image = Image.open(row["full_image_path"]).convert("RGB")

stat = "median"
question = PROMPTS["zero_shot"](stat)

print("Question:", question)
print("Ground truth:", row[stat])

output = ask_model(image, question)
print("Model output:", output)

pred = extract_number(output)
print("Predicted number:", pred)

if pred is not None:
    print("Absolute error:", abs(pred - row[stat]))

In [ ]:
row = df_valid.iloc[0]
image = Image.open(row["full_image_path"]).convert("RGB")
stat = "median"
gt = row[stat]

for prompt_type, prompt_fn in PROMPTS.items():
    question = prompt_fn(stat)
    output = ask_model(image, question)
    pred = extract_number(output)

    print("PROMPT TYPE:", prompt_type)
    print("QUESTION:", question)
    print("GROUND TRUTH:", gt)
    print("MODEL OUTPUT:", output)
    print("PREDICTED:", pred)
    print("ABS ERROR:", None if pred is None else abs(pred - gt))
    print("-" * 80)

In [ ]:
import pandas as pd

sample = df_valid.head(3).copy()
stat_to_test = "median"

results = []

for row_idx, row in sample.iterrows():
    image = Image.open(row["full_image_path"]).convert("RGB")
    gt = row[stat_to_test]

    for prompt_type, prompt_fn in PROMPTS.items():
        question = prompt_fn(stat_to_test)

        try:
            output = ask_model(image, question)
            pred = extract_number(output)
        except Exception as e:
            output = f"ERROR: {e}"
            pred = None

        error = None if pred is None else abs(pred - gt)

        results.append({
            "row_idx": row_idx,
            "image_file": row["image_file"],
            "stat": stat_to_test,
            "ground_truth": gt,
            "prompt_type": prompt_type,
            "question": question,
            "model_output": output,
            "predicted": pred,
            "absolute_error": error,
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
for _, r in results_df.iterrows():
    print("IMAGE:", r["image_file"])
    print("PROMPT:", r["prompt_type"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("MODEL OUTPUT:", r["model_output"])
    print("PREDICTED:", r["predicted"])
    print("ABS ERROR:", r["absolute_error"])
    print("-" * 80)

In [ ]:
summary = results_df.groupby("prompt_type")["absolute_error"].mean()
print(summary)